# Combined clDice Experiment: P1 Memory Test + Real Prediction Eval + Combined Training Loss

本 Notebook 包含三部分：
1. **P1**：官方 `Skeletonize` 模块在不同 patch size (64³/96³/128³) 下的可微反向传播显存压力测试。
2. **真实预测离线评估**：将 `PRED_DIR` 指向 nnU-Net 基线预测输出，评估软骨架拓扑指标。
3. **组合训练 Loss**：主分支 DiceCE 在完整 `[40,256,224]` patch 上，辅助 clDice 在随机 crop 的 `64³` 前景 patch 上，权重 λ=0.1。

> 注意：Cell 2 中的 `PRED_DIR` 必须指向你的真实 nnU-Net 预测目录；若留 None 则 fallback 到模拟概率图。

In [1]:
# Cell 1：环境安装与依赖导入
import os, sys, inspect, glob, gc, random, warnings, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import nibabel as nib
except ImportError:
    !pip -q install nibabel
    import nibabel as nib

try:
    from scipy.ndimage import gaussian_filter, label, binary_dilation, distance_transform_edt
except ImportError:
    !pip -q install scipy
    from scipy.ndimage import gaussian_filter, label, binary_dilation, distance_transform_edt

import matplotlib.pyplot as plt
from tqdm import tqdm

warnings.filterwarnings('ignore')

REPO_DIR = '/tmp/skel_repo'
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/martinmenten/skeletonization-for-gradient-based-optimization.git {REPO_DIR}
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from skeletonize import Skeletonize

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('Skeletonize signature:', inspect.signature(Skeletonize.__init__))

Device: cuda:0
Skeletonize signature: (self, probabilistic=True, beta=0.33, tau=1.0, simple_point_detection='Boolean', num_iter=5)


In [2]:
# Cell 2：路径配置与全局超参数
# ==================== 已按你的实际路径修改 ====================

HARD_SKEL_DIR = '/kaggle/input/datasets/xztkaltsit/nnu-net-pre-nrrd/Results_C/thick_skel_labels_expC'
GT_DIR = '/kaggle/input/datasets/xztkaltsit/nnu-net-train-data/nnUNet_preprocessed/Dataset001_MRCP/gt_segmentations'

# 基线模型预测结果目录（12 个 .nii 文件）
PRED_DIR = '/kaggle/input/datasets/xztkaltsit/nnu-net-pre-nrrd'

# 基线模型权重（暂未在 Notebook 中使用，可留作后续使用）
MODEL_PATH = '/kaggle/input/datasets/xztkaltsit/nnu-net-pre-nrrd/checkpoint_best (1).pth'

# 手动指定要评估的 5 个病例（可保证与前 5 个病例相同）
CASE_IDS = [
    "A_1174043", "A_1556353", "A_1628547", "A_1820605", "A_2604999"
]

SAVE_DIR = '/kaggle/working/combined_cldice_experiment'
os.makedirs(SAVE_DIR, exist_ok=True)

SEED = 42
CASE_LIMIT = 5

P1_PATCH_SIZES = [64, 96, 128]
NUM_ITER = 5
BETA = 0.1
TAU = 0.1

EVAL_PATCH_SIZE = 128
EVAL_OVERLAP = 32
SKEL_THRESHOLD = 0.5
TOLERANCE_VOXELS = 1

# 训练参数
MAIN_PATCH_SIZE = (40, 256, 224)   # nnU-Net 3d_fullres 默认 patch
CLDICE_CROP_SIZE = (64, 64, 64)   # 目标 clDice crop；若某维不足则自适应取全维
LAMBDA_CLDICE = 0.1

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

print('SAVE_DIR:', SAVE_DIR)
print('PRED_DIR:', PRED_DIR)
print('CASE_IDS:', CASE_IDS)

SAVE_DIR: /kaggle/working/combined_cldice_experiment
PRED_DIR: /kaggle/input/datasets/xztkaltsit/nnu-net-pre-nrrd
CASE_IDS: ['A_1174043', 'A_1556353', 'A_1628547', 'A_1820605', 'A_2604999']


In [3]:
# Cell 3：工具函数定义
def strip_nii_suffix(filename: str) -> str:
    name = os.path.basename(filename)
    if name.endswith('.nii.gz'): return name[:-7]
    if name.endswith('.nii'): return name[:-4]
    return os.path.splitext(name)[0]

def find_nii(path_without_suffix: str) -> str:
    for ext in ['.nii', '.nii.gz']:
        p = path_without_suffix + ext
        if os.path.exists(p): return p
    raise FileNotFoundError(f'Cannot find {path_without_suffix}.nii[.gz]')

def get_case_ids(hard_skel_dir: str, limit: int = 5):
    files = sorted(glob.glob(os.path.join(hard_skel_dir, 'thick_*.nii*')))
    ids = [strip_nii_suffix(f)[len('thick_'):] for f in files if strip_nii_suffix(f).startswith('thick_')]
    return sorted(ids)[:limit]

def load_case(case_id: str):
    hard = nib.load(find_nii(os.path.join(HARD_SKEL_DIR, f'thick_{case_id}'))).get_fdata() > 0.5
    gt_nii = nib.load(find_nii(os.path.join(GT_DIR, case_id)))
    gt = (gt_nii.get_fdata() > 0.5).astype(np.float32)
    hard = hard.astype(np.uint8)
    if hard.shape != gt.shape:
        raise ValueError(f'Shape mismatch {case_id}: hard={hard.shape}, gt={gt.shape}')
    return {'case_id': case_id, 'hard': hard, 'gt': gt, 'affine': gt_nii.affine}

class OfficialSkeletonization3D(nn.Module):
    def __init__(self, num_iter=5, beta=0.1, tau=0.1, probabilistic=True):
        super().__init__()
        sig = inspect.signature(Skeletonize.__init__)
        vp = sig.parameters.keys()
        kw = {}
        if 'num_iter' in vp: kw['num_iter'] = num_iter
        elif 'iter_num' in vp: kw['iter_num'] = num_iter
        if 'beta' in vp: kw['beta'] = beta
        if 'tau' in vp: kw['tau'] = tau
        elif 'temperature' in vp: kw['temperature'] = tau
        if 'probabilistic' in vp: kw['probabilistic'] = probabilistic
        self.skel = Skeletonize(**kw)
        self.kw = kw
    def forward(self, x): return self.skel(x)

def crop_slices_around_foreground(fg_mask: np.ndarray, patch_size=64, seed=42):
    rng = np.random.default_rng(seed)
    coords = np.argwhere(fg_mask > 0)
    if len(coords) == 0: center = np.array(fg_mask.shape) // 2
    else: center = coords[rng.integers(0, len(coords))]
    slices = []
    for c, dim in zip(center, fg_mask.shape):
        if dim <= patch_size:
            start, end = 0, dim
        else:
            start = max(0, min(int(c) - patch_size // 2, dim - patch_size))
            end = start + patch_size
        slices.append(slice(start, end))
    return tuple(slices)

def make_simulated_probability(gt: np.ndarray, sigma=1.0, low=0.02, high=0.98):
    prob = gaussian_filter(gt.astype(np.float32), sigma=sigma)
    if prob.max() > 0: prob = prob / prob.max()
    return np.clip(prob, low, high).astype(np.float32)

def hard_skel_to_distance_soft_label(hard_skel: np.ndarray, sigma=1.0, radius=2.0):
    hard = hard_skel.astype(bool)
    if hard.sum() == 0: return np.zeros_like(hard_skel, dtype=np.float32)
    dist = distance_transform_edt(~hard)
    soft = np.exp(-(dist ** 2) / (2 * sigma ** 2)).astype(np.float32)
    soft[dist > radius] = 0.0
    return soft

def safe_div(a, b): return float(a) / float(b) if b != 0 else 0.0

def dice_binary(a: np.ndarray, b: np.ndarray):
    a, b = a.astype(bool), b.astype(bool)
    denom = int(a.sum() + b.sum())
    return 1.0 if denom == 0 else 2.0 * int(np.logical_and(a, b).sum()) / denom

def skeleton_precision_recall_tolerant(pred_skel: np.ndarray, target_skel: np.ndarray, tol=1):
    pred, target = pred_skel.astype(bool), target_skel.astype(bool)
    s = np.ones((3, 3, 3), dtype=bool)
    pdil = binary_dilation(pred, structure=s, iterations=tol) if tol > 0 else pred
    tdil = binary_dilation(target, structure=s, iterations=tol) if tol > 0 else target
    prec = safe_div(int(np.logical_and(pred, tdil).sum()), int(pred.sum()))
    rec = safe_div(int(np.logical_and(target, pdil).sum()), int(target.sum()))
    f1 = safe_div(2 * prec * rec, prec + rec) if (prec + rec) > 0 else 0.0
    return prec, rec, f1

def connected_component_stats(binary_skel: np.ndarray):
    vox = int(binary_skel.sum())
    if vox == 0: return 0, 0.0
    lab, n = label(binary_skel.astype(np.uint8), structure=np.ones((3, 3, 3), dtype=np.uint8))
    sizes = np.bincount(lab.ravel())[1:]
    return int(n), float(sizes.max() / vox) if len(sizes) > 0 else 0.0

def summarize_grad(x: torch.Tensor):
    g = x.grad
    if g is None: return {'grad_exists': False, 'grad_nonzero_ratio': 0.0, 'grad_abs_mean': 0.0, 'grad_abs_max': 0.0, 'grad_finite': False}
    a = g.detach().abs()
    return {'grad_exists': True, 'grad_nonzero_ratio': float((a > 0).float().mean()), 'grad_abs_mean': float(a.mean()), 'grad_abs_max': float(a.max()), 'grad_finite': bool(torch.isfinite(g).all())}

def soft_cldice_loss_with_official_skeletonize(pred_prob, target_mask, skel_module, smooth=1.0):
    target_mask = target_mask.detach()
    pred_skel = skel_module(pred_prob)
    with torch.no_grad():
        target_skel = skel_module(target_mask)
    tprec = (torch.sum(pred_skel * target_mask) + smooth) / (torch.sum(pred_skel) + smooth)
    tsens = (torch.sum(target_skel * pred_prob) + smooth) / (torch.sum(target_skel) + smooth)
    cldice = (2.0 * tprec * tsens + smooth) / (tprec + tsens + smooth)
    return 1.0 - cldice, pred_skel, target_skel, {'tprec': tprec.detach(), 'tsens': tsens.detach(), 'cldice': cldice.detach()}

def random_crop_foreground_3d(tensor, mask, crop_size=(64, 64, 64), seed=None):
    if seed is not None: torch.manual_seed(seed); np.random.seed(seed)
    B, C, D, H, W = tensor.shape
    cd, ch, cw = crop_size
    m = mask[0, 0].cpu().numpy() if mask.is_cuda else mask[0, 0].numpy()
    coords = np.argwhere(m > 0.5)
    if len(coords) == 0:
        center = np.array([D, H, W]) // 2
    else:
        center = coords[random.randint(0, len(coords) - 1)]
    slices = []
    for c, dim, cs in zip(center, (D, H, W), crop_size):
        if dim <= cs:
            start, end = 0, dim
        else:
            start = max(0, min(int(c) - cs // 2, dim - cs))
            end = start + cs
        slices.append(slice(start, end))
    slc = (slice(None), slice(None)) + tuple(slices)
    return tensor[slc], mask[slc], slices

@torch.no_grad()
def skeletonize_in_patches_eval_only(model, tensor_5d, patch_size=128, overlap=32):
    assert tensor_5d.ndim == 5 and tensor_5d.shape[0] == 1 and tensor_5d.shape[1] == 1
    model.eval()
    dev = tensor_5d.device
    tensor_5d = tensor_5d.to(dev)
    _, _, D, H, W = tensor_5d.shape
    stride = patch_size - overlap
    out = torch.zeros((1, 1, D, H, W), dtype=torch.float32, device='cpu')
    d_steps = list(range(0, max(D - overlap, 1), stride))
    h_steps = list(range(0, max(H - overlap, 1), stride))
    w_steps = list(range(0, max(W - overlap, 1), stride))
    for ds in d_steps:
        de = min(ds + patch_size, D)
        ds = max(0, de - patch_size) if (de - ds) < patch_size and D >= patch_size else ds
        for hs in h_steps:
            he = min(hs + patch_size, H)
            hs = max(0, he - patch_size) if (he - hs) < patch_size and H >= patch_size else hs
            for ws in w_steps:
                we = min(ws + patch_size, W)
                ws = max(0, we - patch_size) if (we - ws) < patch_size and W >= patch_size else ws
                patch = tensor_5d[:, :, ds:de, hs:he, ws:we].contiguous()
                skel_patch = model(patch).detach().cpu()
                d0 = 0 if ds == 0 else overlap // 2
                h0 = 0 if hs == 0 else overlap // 2
                w0 = 0 if ws == 0 else overlap // 2
                d1 = skel_patch.shape[2] if de == D else skel_patch.shape[2] - overlap // 2
                h1 = skel_patch.shape[3] if he == H else skel_patch.shape[3] - overlap // 2
                w1 = skel_patch.shape[4] if we == W else skel_patch.shape[4] - overlap // 2
                od0, oh0, ow0 = ds + d0, hs + h0, ws + w0
                core = skel_patch[:, :, d0:d1, h0:h1, w0:w1]
                out[:, :, od0:od0 + (d1 - d0), oh0:oh0 + (h1 - h0), ow0:ow0 + (w1 - w0)] = torch.maximum(
                    out[:, :, od0:od0 + (d1 - d0), oh0:oh0 + (h1 - h0), ow0:ow0 + (w1 - w0)], core
                )
                del patch, skel_patch, core
                torch.cuda.empty_cache()
    return out

case_ids = get_case_ids(HARD_SKEL_DIR, CASE_LIMIT)
case_info = {cid: load_case(cid) for cid in case_ids}
print('Cases:', case_ids)
for cid in case_ids:
    print(cid, 'shape=', case_info[cid]['gt'].shape, 'gt_vox=', int(case_info[cid]['gt'].sum()))

Cases: ['A_1065847', 'A_1093394', 'A_1153975', 'A_1174043', 'A_1286056']
A_1065847 shape= (576, 576, 76) gt_vox= 10902
A_1093394 shape= (528, 528, 108) gt_vox= 16608
A_1153975 shape= (628, 576, 76) gt_vox= 9194
A_1174043 shape= (576, 576, 66) gt_vox= 5171
A_1286056 shape= (528, 528, 108) gt_vox= 4199


In [4]:
# Cell 4：P1 — 可微反向传播显存压力测试 (64³ / 96³ / 128³)
def run_backward_test(case_id, patch_size):
    info = case_info[case_id]
    slc = crop_slices_around_foreground(info['gt'], patch_size=patch_size, seed=SEED)
    gt_patch = info['gt'][slc].astype(np.float32)
    hard_patch = info['hard'][slc].astype(np.float32)
    prob_patch = make_simulated_probability(gt_patch, sigma=1.0)
    
    pred_prob = torch.from_numpy(prob_patch).to(DEVICE).unsqueeze(0).unsqueeze(0).requires_grad_(True)
    target_mask = torch.from_numpy(gt_patch).to(DEVICE).unsqueeze(0).unsqueeze(0)
    hard_dist = torch.from_numpy(hard_skel_to_distance_soft_label(hard_patch)).to(DEVICE).unsqueeze(0).unsqueeze(0)
    
    skel_mod = OfficialSkeletonization3D(num_iter=NUM_ITER, beta=BETA, tau=TAU).to(DEVICE)
    skel_mod.train()
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    
    t0 = time.time()
    cl_loss, pred_skel, _, logs = soft_cldice_loss_with_official_skeletonize(pred_prob, target_mask, skel_mod, smooth=1.0)
    aux_mse = F.mse_loss(pred_skel, hard_dist)
    loss = cl_loss + 0.05 * aux_mse
    loss.backward()
    elapsed = time.time() - t0
    peak = torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else 0.0
    
    gs = summarize_grad(pred_prob)
    result = {
        'case': case_id, 'patch_shape': tuple(gt_patch.shape), 'status': 'ok',
        'loss': float(loss.detach().cpu()), 'cl_loss': float(cl_loss.detach().cpu()),
        'tprec': float(logs['tprec'].cpu()), 'tsens': float(logs['tsens'].cpu()),
        'cldice': float(logs['cldice'].cpu()), 'peak_mem_gb': round(peak, 3),
        'elapsed_sec': round(elapsed, 2), **gs
    }
    
    del pred_prob, target_mask, hard_dist, pred_skel, loss, skel_mod
    gc.collect(); torch.cuda.empty_cache()
    return result

rows = []
for ps in tqdm(P1_PATCH_SIZES, desc='P1 patch sizes'):
    print(f'\n>>> Testing patch_size={ps}')
    try:
        r = run_backward_test(case_ids[0], ps)
        rows.append(r)
        tag = '✅ 通过' if r['grad_exists'] and r['grad_finite'] and r['grad_abs_max'] > 0 else '❌ 异常'
        print(f"{tag} | loss={r['loss']:.4f} | peak_mem={r['peak_mem_gb']:.2f}GB | grad_max={r['grad_abs_max']:.6f}")
    except torch.cuda.OutOfMemoryError as e:
        print(f'❌ OOM at {ps}³: {e}')
        rows.append({'patch_shape': f'{ps}^3', 'status': 'cuda_oom', 'error': str(e)[:200]})
        gc.collect(); torch.cuda.empty_cache()
    except Exception as e:
        print(f'❌ Error at {ps}³: {type(e).__name__}: {e}')
        rows.append({'patch_shape': f'{ps}^3', 'status': 'error', 'error': f'{type(e).__name__}: {str(e)[:200]}'})
        gc.collect(); torch.cuda.empty_cache()

df_p1 = pd.DataFrame(rows)
df_p1.to_csv(os.path.join(SAVE_DIR, 'p1_memory_test.csv'), index=False, encoding='utf-8-sig')
print('\n=== P1 汇总 ===')
print(df_p1[['patch_shape', 'status', 'peak_mem_gb', 'cldice', 'grad_abs_max']].to_string(index=False))

P1 patch sizes:   0%|          | 0/3 [00:00<?, ?it/s]


>>> Testing patch_size=64


P1 patch sizes:  33%|███▎      | 1/3 [00:03<00:06,  3.42s/it]

✅ 通过 | loss=0.0678 | peak_mem=3.16GB | grad_max=0.026650

>>> Testing patch_size=96


P1 patch sizes:  67%|██████▋   | 2/3 [00:05<00:02,  2.86s/it]

✅ 通过 | loss=0.0595 | peak_mem=8.04GB | grad_max=0.018010

>>> Testing patch_size=128


P1 patch sizes: 100%|██████████| 3/3 [00:07<00:00,  2.38s/it]

❌ OOM at 128³: CUDA out of memory. Tried to allocate 2.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 1.81 MiB is free. Including non-PyTorch memory, this process has 14.56 GiB memory in use. Of the allocated memory 11.72 GiB is allocated by PyTorch, and 2.72 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

=== P1 汇总 ===
 patch_shape   status  peak_mem_gb   cldice  grad_abs_max
(64, 64, 64)       ok        3.163 0.932243       0.02665
(96, 96, 76)       ok        8.043 0.940578       0.01801
       128^3 cuda_oom          NaN      NaN           NaN


In [5]:
# Cell 5：真实 nnU-Net 基线预测 — 离线拓扑评估
def load_prediction_probability(case_id, gt_shape):
    if PRED_DIR is None or not os.path.exists(PRED_DIR):
        return make_simulated_probability(case_info[case_id]['gt'], sigma=1.0), 'simulated_from_gt'
    
    candidates = []
    for pfx in ['', 'pred_', 'probability_', 'proba_']:
        for sfx in ['.nii', '.nii.gz']:
            candidates.append(os.path.join(PRED_DIR, f'{pfx}{case_id}{sfx}'))
    
    pred_path = next((p for p in candidates if os.path.exists(p)), None)
    if pred_path is None:
        files = glob.glob(os.path.join(PRED_DIR, f'*{case_id}*.nii*'))
        pred_path = files[0] if files else None
    
    if pred_path is None:
        return make_simulated_probability(case_info[case_id]['gt'], sigma=1.0), 'simulated_from_gt'
    
    arr = nib.load(pred_path).get_fdata().astype(np.float32)
    if arr.shape != gt_shape:
        raise ValueError(f'Prediction shape mismatch: pred={arr.shape}, gt={gt_shape}')
    if arr.min() < 0 or arr.max() > 1:
        arr = 1.0 / (1.0 + np.exp(-arr))
    return np.clip(arr, 0.0, 1.0).astype(np.float32), pred_path

def run_offline_eval(case_ids_to_run, patch_size=128, overlap=32):
    model = OfficialSkeletonization3D(num_iter=NUM_ITER, beta=BETA, tau=TAU).to(DEVICE)
    records = []
    for cid in tqdm(case_ids_to_run, desc='Offline eval'):
        info = case_info[cid]
        gt = info['gt'].astype(np.float32)
        hard = info['hard'].astype(np.uint8)
        
        pred_prob_np, src = load_prediction_probability(cid, gt.shape)
        pred_prob = torch.from_numpy(pred_prob_np).to(DEVICE).unsqueeze(0).unsqueeze(0)
        
        pred_skel_cpu = skeletonize_in_patches_eval_only(model, pred_prob, patch_size=patch_size, overlap=overlap)
        pred_skel_np = pred_skel_cpu.squeeze().numpy().astype(np.float32)
        pred_skel_bin = (pred_skel_np > SKEL_THRESHOLD).astype(np.uint8)
        
        hard_soft = hard_skel_to_distance_soft_label(hard, sigma=1.0, radius=2.0)
        soft_mse = float(np.mean((pred_skel_np - hard_soft) ** 2))
        
        hv, sv = int(hard.sum()), int(pred_skel_bin.sum())
        exact_dice = dice_binary(pred_skel_bin, hard)
        sp, sr, tf1 = skeleton_precision_recall_tolerant(pred_skel_bin, hard, tol=TOLERANCE_VOXELS)
        n_comp, longest = connected_component_stats(pred_skel_bin)
        
        rec = {
            'case': cid, 'pred_source': str(src),
            'hard_vox': hv, 'soft_vox': sv,
            'volume_diff_ratio': round(safe_div(sv - hv, hv), 4),
            'exact_dice': round(exact_dice, 4),
            f'skel_precision_tol{TOLERANCE_VOXELS}': round(sp, 4),
            f'skel_recall_tol{TOLERANCE_VOXELS}': round(sr, 4),
            f'tolerance_f1_tol{TOLERANCE_VOXELS}': round(tf1, 4),
            'soft_components': n_comp, 'soft_longest_ratio': round(longest, 4),
            'soft_mse_to_hard_dist': soft_mse
        }
        records.append(rec)
        print(pd.Series(rec).to_string())
        
        bg = pred_prob_np.max(axis=0)
        fig, axes = plt.subplots(1, 3, figsize=(16, 5))
        axes[0].imshow(bg, cmap='gray', aspect='auto'); axes[0].set_title(f'{cid} | Pred Prob MIP'); axes[0].axis('off')
        axes[1].imshow(bg, cmap='gray', aspect='auto'); axes[1].imshow(np.ma.masked_where(hard.max(axis=0)==0, hard.max(axis=0)), alpha=0.85); axes[1].set_title('Hard Skeleton'); axes[1].axis('off')
        axes[2].imshow(bg, cmap='gray', aspect='auto'); axes[2].imshow(np.ma.masked_where(pred_skel_bin.max(axis=0)==0, pred_skel_bin.max(axis=0)), alpha=0.85); axes[2].set_title('Soft Skeleton'); axes[2].axis('off')
        plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, f'{cid}_offline_eval.png'), dpi=120); plt.close()
        
        del pred_prob, pred_skel_cpu; gc.collect(); torch.cuda.empty_cache()
    
    df = pd.DataFrame(records)
    df.to_csv(os.path.join(SAVE_DIR, 'offline_topology_metrics.csv'), index=False, encoding='utf-8-sig')
    return df

offline_df = run_offline_eval(case_ids[:5], patch_size=EVAL_PATCH_SIZE, overlap=EVAL_OVERLAP)

print('\n=== 离线评估汇总 ===')
print(offline_df[['case', 'pred_source', 'exact_dice', 'tolerance_f1_tol1', 'soft_components', 'soft_longest_ratio']].to_string(index=False))

Offline eval:   0%|          | 0/5 [00:00<?, ?it/s]

case                             A_1065847
pred_source              simulated_from_gt
hard_vox                               200
soft_vox                               242
volume_diff_ratio                     0.21
exact_dice                          0.2534
skel_precision_tol1                 0.8512
skel_recall_tol1                      0.95
tolerance_f1_tol1                   0.8979
soft_components                          2
soft_longest_ratio                  0.9628
soft_mse_to_hard_dist             0.000026


Offline eval:  20%|██        | 1/5 [00:23<01:33, 23.44s/it]

case                             A_1093394
pred_source              simulated_from_gt
hard_vox                               171
soft_vox                               188
volume_diff_ratio                   0.0994
exact_dice                          0.2953
skel_precision_tol1                 0.8883
skel_recall_tol1                    0.9357
tolerance_f1_tol1                   0.9114
soft_components                          2
soft_longest_ratio                  0.9043
soft_mse_to_hard_dist             0.000018


Offline eval:  40%|████      | 2/5 [00:53<01:21, 27.14s/it]

...

case                                                             A_1174043
pred_source              /kaggle/input/datasets/xztkaltsit/nnu-net-pre-...
hard_vox                                                               157
soft_vox                                                               135
volume_diff_ratio                                                  -0.1401
exact_dice                                                          0.3082
skel_precision_tol1                                                 0.9333
skel_recall_tol1                                                    0.7898
tolerance_f1_tol1                                                   0.8556
soft_components                                                          4
soft_longest_ratio                                                  0.8074
soft_mse_to_hard_dist                                             0.000023


Offline eval:  80%|████████  | 4/5 [01:40<00:24, 24.52s/it]

case                             A_1286056
pred_source              simulated_from_gt
hard_vox                               192
soft_vox                               199
volume_diff_ratio                   0.0365
exact_dice                          0.3478
skel_precision_tol1                 0.9447
skel_recall_tol1                    0.9792
tolerance_f1_tol1                   0.9616
soft_components                          1
soft_longest_ratio                     1.0
soft_mse_to_hard_dist              0.00002


Offline eval: 100%|██████████| 5/5 [02:10<00:00, 26.06s/it]


=== 离线评估汇总 ===
     case                                                      pred_source  exact_dice  tolerance_f1_tol1  soft_components  soft_longest_ratio
A_1065847                                                simulated_from_gt      0.2534             0.8979                2              0.9628
A_1093394                                                simulated_from_gt      0.2953             0.9114                2              0.9043
A_1153975                                                simulated_from_gt      0.3000             0.9467                1              1.0000
A_1174043 /kaggle/input/datasets/xztkaltsit/nnu-net-pre-nrrd/A_1174043.nii      0.3082             0.8556                4              0.8074
A_1286056                                                simulated_from_gt      0.3478             0.9616                1              1.0000


In [6]:
# Cell 6：组合训练 Loss（DiceCE + 随机 Crop 的 clDice）
def dice_loss(pred_prob, target, smooth=1.0):
    pred_prob = pred_prob.contiguous()
    target = target.contiguous()
    intersection = (pred_prob * target).sum(dim=(2, 3, 4))
    return 1.0 - (2.0 * intersection + smooth) / (pred_prob.sum(dim=(2, 3, 4)) + target.sum(dim=(2, 3, 4)) + smooth)

def dice_ce_loss(logits, target, dice_weight=1.0, ce_weight=1.0):
    if logits.shape[1] == 1:
        pred_prob = torch.sigmoid(logits)
        bce = F.binary_cross_entropy_with_logits(logits, target, reduction='mean')
    else:
        pred_prob = torch.softmax(logits, dim=1)[:, 1:2]
        bce = F.cross_entropy(logits, target.squeeze(1).long(), reduction='mean')
    dloss = dice_loss(pred_prob, target).mean()
    return dice_weight * dloss + ce_weight * bce, {'dice_loss': dloss.detach(), 'ce_loss': bce.detach()}

def combined_training_loss(logits, target, skel_module, lambda_cldice=0.1, crop_size=(64, 64, 64), smooth=1.0):
    main_loss, main_logs = dice_ce_loss(logits, target)
    
    prob_full = torch.sigmoid(logits) if logits.shape[1] == 1 else torch.softmax(logits, dim=1)[:, 1:2]
    crop_prob, crop_target, _ = random_crop_foreground_3d(prob_full, target, crop_size=crop_size)
    
    if min(crop_prob.shape[2:]) == 0:
        return main_loss, {'main_loss': main_loss.detach(), 'cldice_loss': torch.tensor(0.0), 'total_loss': main_loss.detach()}
    
    cl_loss, _, _, cl_logs = soft_cldice_loss_with_official_skeletonize(crop_prob, crop_target, skel_module, smooth=smooth)
    total_loss = main_loss + lambda_cldice * cl_loss
    
    logs = {
        'main_loss': main_loss.detach(),
        'dice_loss': main_logs['dice_loss'],
        'ce_loss': main_logs['ce_loss'],
        'cldice_loss': cl_loss.detach(),
        'cldice_crop_shape': tuple(crop_prob.shape[2:]),
        'cldice_tprec': cl_logs['tprec'],
        'cldice_tsens': cl_logs['tsens'],
        'cldice_score': cl_logs['cldice'],
        'total_loss': total_loss.detach()
    }
    return total_loss, logs

print('组合 Loss 函数定义完成。')
print('说明：主分支 DiceCE 在完整 patch 上；辅助 clDice 在随机 crop 的前景区域上。')
print(f'若 patch D={MAIN_PATCH_SIZE[0]} < 64，则 clDice crop 会自动取 D 全维，实际 crop 为 (40,64,64)。')

组合 Loss 函数定义完成。
说明：主分支 DiceCE 在完整 patch 上；辅助 clDice 在随机 crop 的前景区域上。
若 patch D=40 < 64，则 clDice crop 会自动取 D 全维，实际 crop 为 (40,64,64)。


In [7]:
# Cell 7：模拟训练步 —— 验证组合 Loss 的 forward / backward
def simulate_training_step():
    B = 1
    C = 2  # 二分类 logits
    D, H, W = MAIN_PATCH_SIZE
    
    logits = torch.randn(B, C, D, H, W, device=DEVICE, requires_grad=True)
    target = torch.zeros(B, 1, D, H, W, device=DEVICE)
    target[:, :, D//3:2*D//3, H//4:3*H//4, W//4:3*W//4] = 1.0
    
    skel_mod = OfficialSkeletonization3D(num_iter=NUM_ITER, beta=BETA, tau=TAU).to(DEVICE)
    skel_mod.train()
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    
    t0 = time.time()
    total_loss, logs = combined_training_loss(
        logits, target, skel_mod, lambda_cldice=LAMBDA_CLDICE, crop_size=CLDICE_CROP_SIZE
    )
    total_loss.backward()
    elapsed = time.time() - t0
    
    peak = torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else 0.0
    grad_ok = logits.grad is not None and torch.isfinite(logits.grad).all() and (logits.grad.abs().max() > 0)
    
    print(f'模拟训练步完成 ({elapsed:.2f}s)')
    print(f'主 patch: {logits.shape} | clDice crop: {logs["cldice_crop_shape"]}')
    print(f'  main_loss (DiceCE): {logs["main_loss"]:.4f}')
    print(f'  cldice_loss:        {logs["cldice_loss"]:.4f}')
    print(f'  total_loss:         {logs["total_loss"]:.4f}')
    print(f'  peak_mem:           {peak:.2f} GB')
    print(f'  梯度回传正常:       {grad_ok}')
    
    del logits, target, skel_mod; gc.collect(); torch.cuda.empty_cache()

simulate_training_step()

模拟训练步完成 (1.40s)
主 patch: torch.Size([1, 2, 40, 256, 224]) | clDice crop: (40, 64, 64)
  main_loss (DiceCE): 1.7629
  cldice_loss:        0.2734
  total_loss:         1.7903
  peak_mem:           2.11 GB
  梯度回传正常:       True


In [12]:
# Cell 8: 拓扑Betti数对比 — GT Hard vs 最佳Skeleton(Boolean) vs 原代码默认Boolean
# 依赖前面已定义的: case_ids, case_info, DEVICE, SAVE_DIR, EVAL_PATCH_SIZE, EVAL_OVERLAP
#                    SKEL_THRESHOLD, make_simulated_probability, OfficialSkeletonization3D
#                    skeletonize_in_patches_eval_only

try:
    from skimage.measure import euler_number
except ImportError:
    !pip -q install scikit-image
    from skimage.measure import euler_number


def compute_betti_numbers_3d(binary_vol):
    """β0, β1, #points。假设骨架无空腔(β2=0)，β1 = β0 - χ。"""
    binary_vol = binary_vol.astype(bool)
    n_points = int(binary_vol.sum())
    if n_points == 0:
        return 0, 0, 0
    labeled, n_components = label(
        binary_vol.astype(np.uint8),
        structure=np.ones((3, 3, 3), dtype=np.uint8)
    )
    beta0 = int(n_components)
    chi = euler_number(binary_vol, connectivity=3)
    beta1 = beta0 - chi
    return beta0, beta1, n_points


# Skeletonize 参数搜索 Wrapper（仅Boolean）
class SkeletonizeParamSearch(nn.Module):
    def __init__(self, beta=0.33, tau=1.0, num_iter=5, probabilistic=True):
        super().__init__()
        self.skel = Skeletonize(
            probabilistic=probabilistic, beta=beta, tau=tau,
            simple_point_detection='Boolean', num_iter=num_iter
        )
    def forward(self, x):
        return self.skel(x)


# 精简参数空间（6组，只改 beta/tau/iter，固定Boolean）
param_candidates = [
    {'beta': 0.1,  'tau': 0.1,  'num_iter': 3},
    {'beta': 0.1,  'tau': 0.1,  'num_iter': 5},   # 原代码默认
    {'beta': 0.1,  'tau': 0.1,  'num_iter': 10},
    {'beta': 0.33, 'tau': 1.0,  'num_iter': 5},   # 论文默认
    {'beta': 0.33, 'tau': 0.5,  'num_iter': 5},
    {'beta': 0.5,  'tau': 0.5,  'num_iter': 5},
]

records = []

for cid in tqdm(case_ids, desc='Cases'):
    info = case_info[cid]
    gt_hard = info['hard'].astype(np.uint8)

    # GT
    gt_b0, gt_b1, gt_pts = compute_betti_numbers_3d(gt_hard)

    # 统一用GT生成的模拟概率图
    prob = make_simulated_probability(info['gt'], sigma=1.0)
    prob_tensor = torch.from_numpy(prob).to(DEVICE).unsqueeze(0).unsqueeze(0)

    # 原代码默认 Boolean (beta=0.1, tau=0.1, iter=5)
    euler_mod = OfficialSkeletonization3D(
        num_iter=5, beta=0.1, tau=0.1, probabilistic=True
    ).to(DEVICE)
    euler_skel = skeletonize_in_patches_eval_only(
        euler_mod, prob_tensor,
        patch_size=EVAL_PATCH_SIZE, overlap=EVAL_OVERLAP
    )
    euler_bin = (euler_skel.squeeze().numpy() > SKEL_THRESHOLD).astype(np.uint8)
    euler_b0, euler_b1, euler_pts = compute_betti_numbers_3d(euler_bin)
    del euler_mod, euler_skel

    # Skeleton 参数搜索（Boolean only）
    best_error = float('inf')
    best_cfg = None
    best_sk_b0 = best_sk_b1 = best_sk_pts = None

    for cfg in tqdm(param_candidates, desc=f'{cid} params', leave=False):
        sk_mod = SkeletonizeParamSearch(
            beta=cfg['beta'], tau=cfg['tau'], num_iter=cfg['num_iter']
        ).to(DEVICE)

        sk_skel = skeletonize_in_patches_eval_only(
            sk_mod, prob_tensor,
            patch_size=EVAL_PATCH_SIZE, overlap=EVAL_OVERLAP
        )
        sk_bin = (sk_skel.squeeze().numpy() > SKEL_THRESHOLD).astype(np.uint8)
        sk_b0, sk_b1, sk_pts = compute_betti_numbers_3d(sk_bin)

        err = abs(sk_b0 - gt_b0) + abs(sk_b1 - gt_b1)
        if err < best_error:
            best_error = err
            best_cfg = cfg.copy()
            best_sk_b0, best_sk_b1, best_sk_pts = sk_b0, sk_b1, sk_pts

        del sk_mod, sk_skel
        gc.collect()
        torch.cuda.empty_cache()

    rec = {
        'case': cid,
        'GT_points': gt_pts, 'GT_beta0': gt_b0, 'GT_beta1': gt_b1,
        'SK_best_beta': best_cfg['beta'],
        'SK_best_tau': best_cfg['tau'],
        'SK_best_iter': best_cfg['num_iter'],
        'SK_points': best_sk_pts,
        'SK_beta0': best_sk_b0, 'SK_beta1': best_sk_b1,
        'SK_beta0_error': abs(best_sk_b0 - gt_b0),
        'SK_beta1_error': abs(best_sk_b1 - gt_b1),
        'EB_points': euler_pts,
        'EB_beta0': euler_b0, 'EB_beta1': euler_b1,
        'EB_beta0_error': abs(euler_b0 - gt_b0),
        'EB_beta1_error': abs(euler_b1 - gt_b1),
    }
    records.append(rec)

    print(f"\n[{cid}] GT: β0={gt_b0}, β1={gt_b1}, pts={gt_pts}")
    print(f"  Best Skeleton(Boolean) {best_cfg}: β0={best_sk_b0}, β1={best_sk_b1}, "
          f"pts={best_sk_pts}, err=({rec['SK_beta0_error']},{rec['SK_beta1_error']})")
    print(f"  Default Boolean:       β0={euler_b0}, β1={euler_b1}, "
          f"pts={euler_pts}, err=({rec['EB_beta0_error']},{rec['EB_beta1_error']})")

df_betti = pd.DataFrame(records)
csv_path = os.path.join(SAVE_DIR, 'topology_betti_comparison.csv')
df_betti.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f"\n✅ CSV saved: {csv_path}")
print(df_betti.to_string(index=False))

Cases:  20%|██        | 1/5 [02:04<08:16, 124.15s/it]          


[A_1065847] GT: β0=1, β1=0, pts=200
  Best Skeleton(Boolean) {'beta': 0.1, 'tau': 0.1, 'num_iter': 3}: β0=1, β1=-1, pts=350, err=(0,1)
  Default Boolean:       β0=3, β1=0, pts=233, err=(2,0)



Cases:  40%|████      | 2/5 [04:46<07:20, 146.70s/it]          


[A_1093394] GT: β0=1, β1=0, pts=171
  Best Skeleton(Boolean) {'beta': 0.1, 'tau': 0.1, 'num_iter': 3}: β0=1, β1=0, pts=518, err=(0,0)
  Default Boolean:       β0=2, β1=0, pts=189, err=(1,0)



Cases:  60%|██████    | 3/5 [07:10<04:51, 145.50s/it]          


[A_1153975] GT: β0=1, β1=0, pts=215
  Best Skeleton(Boolean) {'beta': 0.1, 'tau': 0.1, 'num_iter': 5}: β0=1, β1=0, pts=240, err=(0,0)
  Default Boolean:       β0=1, β1=0, pts=242, err=(0,0)



Cases:  80%|████████  | 4/5 [09:03<02:12, 132.77s/it]          


[A_1174043] GT: β0=1, β1=0, pts=157
  Best Skeleton(Boolean) {'beta': 0.1, 'tau': 0.1, 'num_iter': 3}: β0=1, β1=0, pts=163, err=(0,0)
  Default Boolean:       β0=1, β1=0, pts=165, err=(0,0)



Cases: 100%|██████████| 5/5 [11:46<00:00, 141.24s/it]          


[A_1286056] GT: β0=1, β1=2, pts=192
  Best Skeleton(Boolean) {'beta': 0.1, 'tau': 0.1, 'num_iter': 3}: β0=1, β1=0, pts=190, err=(0,2)
  Default Boolean:       β0=2, β1=0, pts=194, err=(1,2)

✅ CSV saved: /kaggle/working/combined_cldice_experiment/topology_betti_comparison.csv
     case  GT_points  GT_beta0  GT_beta1  SK_best_beta  SK_best_tau  SK_best_iter  SK_points  SK_beta0  SK_beta1  SK_beta0_error  SK_beta1_error  EB_points  EB_beta0  EB_beta1  EB_beta0_error  EB_beta1_error
A_1065847        200         1         0           0.1          0.1             3        350         1        -1               0               1        233         3         0               2               0
A_1093394        171         1         0           0.1          0.1             3        518         1         0               0               0        189         2         0               1               0
A_1153975        215         1         0           0.1          0.1             5        240      

In [13]:
# Cell 9: 拓扑Betti数对比 — GT Hard vs Skeleton(论文默认) vs Boolean(原代码默认)
# 统一参数: iter=5, simple_point_detection='Boolean'

try:
    from skimage.measure import euler_number
except ImportError:
    !pip -q install scikit-image
    from skimage.measure import euler_number


def compute_betti_numbers_3d(binary_vol):
    """计算3D二值体积的 β0、β1 和体素数。
    假设 β2=0（1D骨架无空腔），由 χ = β0 - β1 + β2 得 β1 = β0 - χ。
    """
    binary_vol = binary_vol.astype(bool)
    n_points = int(binary_vol.sum())
    if n_points == 0:
        return 0, 0, 0
    labeled, n_components = label(
        binary_vol.astype(np.uint8),
        structure=np.ones((3, 3, 3), dtype=np.uint8)
    )
    beta0 = int(n_components)
    chi = euler_number(binary_vol, connectivity=3)
    beta1 = beta0 - chi
    return beta0, beta1, n_points


# Skeletonize Wrapper（固定 Boolean）
class SkeletonizeFixed(nn.Module):
    def __init__(self, beta=0.33, tau=1.0, num_iter=5, probabilistic=True):
        super().__init__()
        self.skel = Skeletonize(
            probabilistic=probabilistic, beta=beta, tau=tau,
            simple_point_detection='Boolean', num_iter=num_iter
        )
    def forward(self, x):
        return self.skel(x)


records = []

for cid in tqdm(case_ids, desc='Cases'):
    info = case_info[cid]
    gt_hard = info['hard'].astype(np.uint8)

    # ---------- GT ----------
    gt_b0, gt_b1, gt_pts = compute_betti_numbers_3d(gt_hard)

    # 统一用GT生成的模拟概率图
    prob = make_simulated_probability(info['gt'], sigma=1.0)
    prob_tensor = torch.from_numpy(prob).to(DEVICE).unsqueeze(0).unsqueeze(0)

    # ---------- ② Skeleton 方法 (论文默认: beta=0.33, tau=1.0, iter=5) ----------
    sk_mod = SkeletonizeFixed(beta=0.33, tau=1.0, num_iter=5).to(DEVICE)
    sk_skel = skeletonize_in_patches_eval_only(
        sk_mod, prob_tensor,
        patch_size=EVAL_PATCH_SIZE, overlap=EVAL_OVERLAP
    )
    sk_bin = (sk_skel.squeeze().numpy() > SKEL_THRESHOLD).astype(np.uint8)
    sk_b0, sk_b1, sk_pts = compute_betti_numbers_3d(sk_bin)
    del sk_mod, sk_skel

    # ---------- ③ Boolean 方法 (原代码默认: beta=0.1, tau=0.1, iter=5) ----------
    eb_mod = OfficialSkeletonization3D(
        num_iter=5, beta=0.1, tau=0.1, probabilistic=True
    ).to(DEVICE)
    eb_skel = skeletonize_in_patches_eval_only(
        eb_mod, prob_tensor,
        patch_size=EVAL_PATCH_SIZE, overlap=EVAL_OVERLAP
    )
    eb_bin = (eb_skel.squeeze().numpy() > SKEL_THRESHOLD).astype(np.uint8)
    eb_b0, eb_b1, eb_pts = compute_betti_numbers_3d(eb_bin)
    del eb_mod, eb_skel

    gc.collect()
    torch.cuda.empty_cache()

    rec = {
        'case': cid,
        # GT
        'GT_points': gt_pts,
        'GT_beta0': gt_b0,
        'GT_beta1': gt_b1,
        # Skeleton (论文默认)
        'SK_points': sk_pts,
        'SK_beta0': sk_b0,
        'SK_beta1': sk_b1,
        'SK_beta0_error': abs(sk_b0 - gt_b0),
        'SK_beta1_error': abs(sk_b1 - gt_b1),
        # Boolean (原代码默认)
        'EB_points': eb_pts,
        'EB_beta0': eb_b0,
        'EB_beta1': eb_b1,
        'EB_beta0_error': abs(eb_b0 - gt_b0),
        'EB_beta1_error': abs(eb_b1 - gt_b1),
    }
    records.append(rec)

    print(f"\n[{cid}] GT: β0={gt_b0}, β1={gt_b1}, pts={gt_pts}")
    print(f"  Skeleton(β=0.33,τ=1.0): β0={sk_b0}, β1={sk_b1}, pts={sk_pts}, err=({rec['SK_beta0_error']},{rec['SK_beta1_error']})")
    print(f"  Boolean(β=0.1,τ=0.1):   β0={eb_b0}, β1={eb_b1}, pts={eb_pts}, err=({rec['EB_beta0_error']},{rec['EB_beta1_error']})")

df_betti = pd.DataFrame(records)
csv_path = os.path.join(SAVE_DIR, 'topology_betti_comparison.csv')
df_betti.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f"\n✅ CSV saved: {csv_path}")
print(df_betti.to_string(index=False))

Cases:  20%|██        | 1/5 [00:34<02:17, 34.43s/it]


[A_1065847] GT: β0=1, β1=0, pts=200
  Skeleton(β=0.33,τ=1.0): β0=203, β1=-4, pts=1000, err=(202,4)
  Boolean(β=0.1,τ=0.1):   β0=3, β1=0, pts=236, err=(2,0)


Cases:  40%|████      | 2/5 [01:19<02:01, 40.56s/it]


[A_1093394] GT: β0=1, β1=0, pts=171
  Skeleton(β=0.33,τ=1.0): β0=312, β1=-21, pts=1438, err=(311,21)
  Boolean(β=0.1,τ=0.1):   β0=2, β1=0, pts=184, err=(1,0)


Cases:  60%|██████    | 3/5 [01:59<01:20, 40.24s/it]


[A_1153975] GT: β0=1, β1=0, pts=215
  Skeleton(β=0.33,τ=1.0): β0=267, β1=-3, pts=850, err=(266,3)
  Boolean(β=0.1,τ=0.1):   β0=2, β1=0, pts=234, err=(1,0)


Cases:  80%|████████  | 4/5 [02:30<00:36, 36.65s/it]


[A_1174043] GT: β0=1, β1=0, pts=157
  Skeleton(β=0.33,τ=1.0): β0=189, β1=-3, pts=559, err=(188,3)
  Boolean(β=0.1,τ=0.1):   β0=1, β1=0, pts=164, err=(0,0)


Cases: 100%|██████████| 5/5 [03:15<00:00, 39.02s/it]


[A_1286056] GT: β0=1, β1=2, pts=192
  Skeleton(β=0.33,τ=1.0): β0=315, β1=5, pts=612, err=(314,3)
  Boolean(β=0.1,τ=0.1):   β0=2, β1=0, pts=189, err=(1,2)

✅ CSV saved: /kaggle/working/combined_cldice_experiment/topology_betti_comparison.csv
     case  GT_points  GT_beta0  GT_beta1  SK_points  SK_beta0  SK_beta1  SK_beta0_error  SK_beta1_error  EB_points  EB_beta0  EB_beta1  EB_beta0_error  EB_beta1_error
A_1065847        200         1         0       1000       203        -4             202               4        236         3         0               2               0
A_1093394        171         1         0       1438       312       -21             311              21        184         2         0               1               0
A_1153975        215         1         0        850       267        -3             266               3        234         2         0               1               0
A_1174043        157         1         0        559       189        -3             188   

In [14]:
import shutil
from pathlib import Path

def zip_folder(folder_path: str, output_zip_path: str = None) -> Path:
    """
    将指定文件夹打包为 ZIP 压缩包（保留文件夹本身为顶层目录）

    Parameters
    ----------
    folder_path : str
        要打包的文件夹路径
    output_zip_path : str, optional
        输出 ZIP 文件的完整路径（包括 .zip 后缀）。若为 None，则自动生成到父目录下。

    Returns
    -------
    Path
        生成的 ZIP 文件路径
    """
    folder = Path(folder_path).resolve()
    if not folder.exists():
        raise FileNotFoundError(f"文件夹不存在: {folder}")
    if not folder.is_dir():
        raise NotADirectoryError(f"路径不是目录: {folder}")

    # 确定输出路径
    if output_zip_path is None:
        output_zip_path = folder.parent / f"{folder.name}.zip"
    else:
        output_zip_path = Path(output_zip_path)

    # 使用 shutil.make_archive，base_name 不能包含 .zip 后缀
    base_name = output_zip_path.with_suffix('')
    shutil.make_archive(
        str(base_name),
        'zip',
        root_dir=str(folder.parent),   # 根目录设为父目录
        base_dir=folder.name           # 只打包目标文件夹，保留其名称作为顶层
    )
    print(f"打包成功: {output_zip_path}")
    return output_zip_path

if __name__ == "__main__":
    # 要打包的目标路径
    target_dir = "/kaggle/working/combined_cldice_experiment"
    zip_folder(target_dir)

打包成功: /kaggle/working/combined_cldice_experiment.zip


In [11]:
# 快速测试
for spd in ['Boolean', 'Euler', 'Euler characteristic', 'euler']:
    try:
        s = Skeletonize(simple_point_detection=spd)
        print(f"✅ '{spd}' 可用")
    except Exception as e:
        print(f"❌ '{spd}' 不可用: {e}")

✅ 'Boolean' 可用
❌ 'Euler' 不可用: 
❌ 'Euler characteristic' 不可用: 
❌ 'euler' 不可用: 


## 运行说明

1. **Cell 1~4**：先跑 P1 显存测试。若 64³ 通过、96³/128³ OOM，则训练时 clDice crop 必须 ≤64³。
2. **Cell 2 的 PRED_DIR**：指向你真实的 nnU-Net 基线预测目录后，跑 Cell 5 看真实预测下的拓扑指标。
3. **Cell 6~7**：组合 Loss 模板，确认能 backward 且显存可控后，集成到你的 nnU-Net Trainer。
4. 若 T4 双卡 16GB 下 64³ 仍超显存，将 `CLDICE_CROP_SIZE` 改为 `(40, 48, 48)` 或 `(32, 64, 64)`。